<a href="https://colab.research.google.com/github/sc-godela/ai-wardrobe-stylist/blob/main/notebooks/02_asos_data_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/ai-wardrobe-stylist/data"

Mounted at /content/drive


In [2]:
import pandas as pd

items = pd.read_csv(f"{DATA}/fashion_items.csv")
mw_train = pd.read_csv(f"{DATA}/outfits_MW_train.csv")
mw_test = pd.read_csv(f"{DATA}/outfits_MW_test.csv")
print(items.shape, mw_train.shape, mw_test.shape)
items.head()

(591738, 4) (1226594, 7) (364484, 7)


,item_id,description,image_url,name
0,59681,<ASOSLINEBR>Classic short boots by Ugg<ASOSLIN...,http://images.asos-media.com/inv/media/1/8/6/9...,UGG Classic Short Boots
1,85441,<ASOSLINEBR>Low <a href= /Men/Shoes/Trainers/C...,http://images.asos-media.com/inv/media/1/4/4/5...,Converse Chuck Taylor All Star Core Trainers
2,117490,<ASOSLINEBR>- Gold-finish chain necklace with ...,http://images.asos-media.com/inv/F/1/10/117490...,ASOS Trucker Girl Necklace in the style of Vic...
3,138674,<ASOSLINEBR>- Silver finish necklace with enam...,http://images.asos-media.com/inv/G/1/10/138674...,ASOS Long Guitar Pendant Necklace in the style...
4,161035,<ASOSLINEBR>Sexy platform heel slingback shoes...,http://images.asos-media.com/inv/I/6/218/16103...,ASOS Sexy Slingback Platform Peep Toe Shoe in ...


In [3]:
import requests

mw_ids = set(mw_train["item_id"]) | set(mw_test["item_id"])
mw_items = items[items["item_id"].isin(mw_ids)]
test = mw_items.sample(30, random_state=0)

ok = 0
for url in test["image_url"]:
    if url.startswith("//"):
        url = "https:" + url
    try:
        r = requests.get(url, timeout=10)
        ok += r.status_code == 200 and r.headers.get("content-type", "").startswith("image")
    except Exception:
        pass
print(f"{ok}/30 menswear image links work")

0/30 menswear image links work


In [4]:
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"}

for url in test["image_url"].head(5):
    if url.startswith("//"):
        url = "https:" + url
    try:
        r = requests.get(url, headers=headers, timeout=10)
        print(r.status_code, r.headers.get("content-type"), url)
    except Exception as e:
        print("ERROR", type(e).__name__, url)

503 text/html http://images.asos-media.com/inv/media/9/3/3/7/8347339/peacoat/image1xl.jpg
503 text/html http://images.asos-media.com/inv/media/7/4/8/4/6804847/black09090/image1xl.jpg
503 text/html http://images.asos-media.com/inv/media/7/6/6/1/3161667/grey/image1xl.jpg
503 text/html http://images.asos-media.com/inv/media/7/8/2/8/4758287/black/image1xl.jpg
503 text/html http://images.asos-media.com/inv/media/9/1/4/9/9549419/green/image1xl.jpg


In [5]:
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.asos.com/",
}

for url in test["image_url"].head(5):
    if url.startswith("//"):
        url = "https:" + url
    r = requests.get(url, headers=headers, timeout=10)
    print(r.status_code, url)
    time.sleep(2)

503 http://images.asos-media.com/inv/media/9/3/3/7/8347339/peacoat/image1xl.jpg
503 http://images.asos-media.com/inv/media/7/4/8/4/6804847/black09090/image1xl.jpg
503 http://images.asos-media.com/inv/media/7/6/6/1/3161667/grey/image1xl.jpg
503 http://images.asos-media.com/inv/media/7/8/2/8/4758287/black/image1xl.jpg
503 http://images.asos-media.com/inv/media/9/1/4/9/9549419/green/image1xl.jpg


## Status (work in progress)
**Goal:** check whether ASOS image URLs still work, so we can build
our own CLIP embeddings (ASOS's provided embeddings come from their
private GORDN model, which we can't apply to new photos).

**Results so far:**
- Random sample of 30 menswear image URLs: **0/30 downloaded**.
- All requests return **HTTP 503**, also with browser-like headers
  (User-Agent, Accept, Referer) and 2-second delays.

**Interpretation:** 503 suggests the server is blocking automated
requests, but the images may also no longer exist.

**Next:**
1. Open a URL manually in a browser to tell "blocked" from "deleted".
2. If images are unavailable to scripts → choose a fallback plan
   (we will not try to bypass the site's protection).

In [6]:
print(test["image_url"].iloc[0])

http://images.asos-media.com/inv/media/9/3/3/7/8347339/peacoat/image1xl.jpg


**Browser test:** opening a sampled URL manually shows
"Service unavailable – DNS failure". The image host no longer
resolves, so the images are permanently gone (not just blocked).

**Conclusion:** ASOS outfit structure and text descriptions are
usable; images are not.